# 03 — Gold: Conformed Dimensions

**Phase:** 5 (Sun 17 May afternoon – Mon 18 May)

Builds 13 conformed dimensions in `NEXAMART_GOLD`. Owners per `docs/domain_assignments.md`:

| Dim | SCD | Owner |
|---|---|---|
| dim_date | Static + role-playing | Lead (starter cell below) |
| dim_product | SCD2 | M3 |
| dim_store | SCD1 | M4 |
| dim_customer | SCD2 + identity bridge | M2 (most complex) |
| dim_seller | SCD1 (carries seller_risk_score) | M6 |
| dim_promotion | SCD1 | M5 |
| dim_channel | Static | Lead |
| dim_payment_method | Static + payment_certainty flag | Lead |
| dim_delivery_method | Static + is_platform_controlled | Lead |
| dim_listing_condition | Static | M3 |
| dim_return_reason | Static + channel_fault_attribution | Lead |
| dim_step | Static (clickstream funnel) | Lead |
| dim_seller_risk_tier | SCD1 (5 tiers + listing_visibility_impact) | Lead |

## Universal contract (every dim)

1. **Surrogate key** as PK named `<dim>_key` via `surrogate_key()` from `_shared/utils_keys`
2. **Natural key** preserved as a column for traceability
3. **Mandatory anomaly columns** carried from Silver (`anomaly_flag`, `anomaly_reason_code`, `data_quality_status`, `metric_certainty_level`)
4. **`is_active`** boolean (TRUE for current row, also TRUE for static dims)
5. SCD2 dims also carry `valid_from`, `valid_to`, `is_current`, `version_number`

## Setup — Install connector + widgets (Free Edition serverless)

_Brief Section 7.4 specified a Spark-Snowflake Maven JAR; Free Edition replaces this with the pure-Python `snowflake-connector-python`. See `docs/databricks_setup.md`._

In [ ]:
%pip install -q snowflake-connector-python pandas rapidfuzz
# dbutils.library.restartPython()  # SKIP on Free Edition — kernel hangs

In [ ]:
dbutils.widgets.text('sf_account',   'rhxendw-yb24678')
dbutils.widgets.text('sf_user',      'NEXAMART_LEAD')   # change to NEXAMART_M{2..6} for members
dbutils.widgets.text('sf_password',  '')                # paste at notebook run time
dbutils.widgets.text('sf_warehouse', 'NEXAMART_WH')
dbutils.widgets.text('sf_role',      'ACCOUNTADMIN')    # NEXAMART_ENGINEER for members

## Cell 2 — Imports & helpers (used by all dim sections)

In [ ]:
from pyspark.sql import functions as F, Window
import sys
sys.path.append('/Workspace/Repos/<your-org>/nexamart-m1/notebooks/_shared')
from utils_dates     import parse_date, is_parse_failure
from utils_keys      import surrogate_key, anonymous_key
from utils_anomaly   import add_anomaly_columns, flag, flag_date_parse_failure
from utils_snowflake import read_from_snowflake, write_to_snowflake

def read_bronze(t):
    return read_from_snowflake(spark, t, schema='NEXAMART_BRONZE')

def read_silver(t):
    return read_from_snowflake(spark, t, schema='NEXAMART_SILVER')

def write_silver(df, t):
    n = write_to_snowflake(df, t, schema='NEXAMART_SILVER')
    print(f'Wrote silver.{t} ({n} rows)')

def write_gold(df, t):
    n = write_to_snowflake(df, t, schema='NEXAMART_GOLD')
    print(f'Wrote gold.{t} ({n} rows)')

## dim_date (Lead) — starter implementation

Static + role-playing. One row per calendar day from 2024-01-01 to 2024-12-31. Includes `is_campaign_day`, `is_weekend`, `is_public_holiday`.

In [ ]:
from pyspark.sql import Row
from datetime import date, timedelta   # fix: dim_date uses date()/timedelta below

# Indian public holidays 2024 — minimal list; extend as needed
PUBLIC_HOLIDAYS_2024 = {
    date(2024, 1, 26),  # Republic Day
    date(2024, 3, 25),  # Holi
    date(2024, 4, 11),  # Eid-ul-Fitr
    date(2024, 8, 15),  # Independence Day
    date(2024, 10, 2),  # Gandhi Jayanti
    date(2024, 11, 1),  # Diwali
    date(2024, 12, 25), # Christmas
}

CAMPAIGN_START = date(2024, 8, 8)
CAMPAIGN_END   = date(2024, 8, 28)

rows = []
d = date(2024, 1, 1)
while d <= date(2024, 12, 31):
    rows.append(Row(
        date_iso=d.isoformat(),
        year=d.year,
        month=d.month,
        day=d.day,
        day_of_week=d.isoweekday(),  # 1=Mon, 7=Sun
        is_weekend=(d.isoweekday() >= 6),
        is_public_holiday=(d in PUBLIC_HOLIDAYS_2024),
        is_campaign_day=(CAMPAIGN_START <= d <= CAMPAIGN_END),
        campaign_phase=(
            'BASELINE'      if d < date(2024, 8, 1) else
            'PRE_RAMP'      if d < CAMPAIGN_START else
            'CAMPAIGN'      if d <= CAMPAIGN_END else
            'POST_CAMPAIGN' if d <= date(2024, 9, 14) else
            'OUT_OF_SCOPE'
        ),
    ))
    d += timedelta(days=1)

dim_date = spark.createDataFrame(rows)
dim_date = dim_date.withColumn('date_key', surrogate_key(F.col('date_iso')))
dim_date = add_anomaly_columns(dim_date)

write_gold(dim_date, 'dim_date')
print('dim_date rows = %d (expect 366; 2024 is a leap year)' % dim_date.count())

## dim_channel (Lead, static)

Five channels: STORE / EC / MARKETPLACE_FBN / MARKETPLACE_SF / NEXALOCAL.

In [ ]:
channels = [
    Row(channel_code='STORE',           channel_name='Physical Store',          is_platform_controlled=True),
    Row(channel_code='EC',              channel_name='Ecommerce',                is_platform_controlled=True),
    Row(channel_code='MARKETPLACE_FBN', channel_name='Marketplace (FBN)',        is_platform_controlled=True),
    Row(channel_code='MARKETPLACE_SF',  channel_name='Marketplace (Seller-fulfilled)', is_platform_controlled=False),
    Row(channel_code='NEXALOCAL',       channel_name='NexaLocal Classifieds',    is_platform_controlled=False),
]
dim_channel = spark.createDataFrame(channels)
dim_channel = dim_channel.withColumn('channel_key', surrogate_key(F.col('channel_code')))
dim_channel = add_anomaly_columns(dim_channel)
write_gold(dim_channel, 'dim_channel')

## dim_payment_method (Lead, static + payment_certainty)

**TODO:** Read `pg_instrument_types` from Silver/Bronze; add `payment_certainty` flag (CONFIRMED for CARD/UPI/NETBANKING; INFERRED for COD).

In [ ]:
# dim_payment_method (Lead, static) — from pg_instrument_types
# payment_certainty: COD settles on delivery (INFERRED at order time); prepaid rails are CONFIRMED.
pm = read_bronze('pg_instrument_types')   # type_code, type_name, provider
pm = (pm
      .withColumn('payment_certainty',
                  F.when((F.upper(F.col('type_code')).contains('COD')) |
                         (F.lower(F.col('type_name')).contains('cash')) |
                         (F.lower(F.col('type_name')).contains('delivery')),
                         F.lit('INFERRED')).otherwise(F.lit('CONFIRMED')))
      .withColumn('is_prepaid', F.col('payment_certainty') == F.lit('CONFIRMED'))
      .withColumn('payment_method_key', surrogate_key(F.col('type_code')))
      .withColumn('is_active', F.lit(True)))
dim_payment_method = add_anomaly_columns(pm)
write_gold(dim_payment_method, 'dim_payment_method')
print('dim_payment_method rows = %d' % dim_payment_method.count())

## dim_delivery_method (Lead, static + is_platform_controlled)

**TODO:** From `ec_delivery_methods`. Add `is_platform_controlled` (TRUE for Standard/Express/BOPIS via NexaMart fleet; FALSE for marketplace seller-shipped).

In [ ]:
# dim_delivery_method (Lead, static) — from ec_delivery_methods
# is_platform_controlled: NexaMart-fleet methods (Standard/Express/BOPIS) TRUE; seller-shipped FALSE.
dm = read_bronze('ec_delivery_methods')   # method_code, method_name, est_days_min, est_days_max, base_fee
SELLER_SHIPPED = ['MARKETPLACE_SELLER', 'NL_SELLER_HANDOFF', 'SELLER', 'MARKETPLACE']
dm = (dm
      .withColumn('is_platform_controlled',
                  ~F.col('method_code').isin(SELLER_SHIPPED))
      .withColumn('is_bopis', F.upper(F.col('method_code')) == F.lit('BOPIS'))
      .withColumn('delivery_method_key', surrogate_key(F.col('method_code')))
      .withColumn('is_active', F.lit(True)))
dim_delivery_method = add_anomaly_columns(dm)
write_gold(dim_delivery_method, 'dim_delivery_method')
print('dim_delivery_method rows = %d' % dim_delivery_method.count())

## dim_return_reason (M6, static + channel_fault_attribution)

**TODO:** From `rr_return_reasons`. Add `reason_group` (Product Quality / Wrong Item / Change of Mind / Not as Described / Delivery Issue / Seller Issue) and `channel_fault_attribution` (which channel is at fault).

In [ ]:
# TODO M6: dim_return_reason

## dim_step (Lead, static — clickstream funnel)

**TODO:** Defines the customer journey funnel stages. Suggested rows:
- AWARENESS (homepage, search, browse)
- CONSIDERATION (PDP view, compare, wishlist)
- INTENT (add to cart, checkout init)
- CONVERSION (payment, order confirmation)
- POST_PURCHASE (order tracking, support, review)

Plus `sequence_number` and `is_conversion_step`.

In [ ]:
# dim_step (Lead, static — clickstream funnel) — 5 journey stages
steps = [
    Row(step_code='AWARENESS',     step_name='Awareness',     sequence_number=1, is_conversion_step=False),
    Row(step_code='CONSIDERATION', step_name='Consideration', sequence_number=2, is_conversion_step=False),
    Row(step_code='INTENT',        step_name='Intent',        sequence_number=3, is_conversion_step=False),
    Row(step_code='CONVERSION',    step_name='Conversion',    sequence_number=4, is_conversion_step=True),
    Row(step_code='POST_PURCHASE', step_name='Post-Purchase', sequence_number=5, is_conversion_step=False),
]
dim_step = spark.createDataFrame(steps)
dim_step = (dim_step
            .withColumn('step_key', surrogate_key(F.col('step_code')))
            .withColumn('is_active', F.lit(True)))
dim_step = add_anomaly_columns(dim_step)
write_gold(dim_step, 'dim_step')
print('dim_step rows = %d (expect 5)' % dim_step.count())

## dim_seller_risk_tier (M2, SCD1)

**TODO:** 5 tiers per T10 mapping (Verified Trusted, Standard, Under Review, High Risk, Suspended). Each carries `tier_threshold`, `moderation_required` (bool), `listing_visibility_impact` (e.g. 100/100/50/20/0%).

In [ ]:
# TODO M2: dim_seller_risk_tier

## dim_listing_condition (M3, static)

**TODO:** From `pc_condition_codes`. SK on `condition_code`. Add `granularity_group` (NEW / OPEN_BOX / REFURBISHED / USED / DAMAGED).

In [ ]:
# TODO M3: dim_listing_condition

## dim_promotion (M5, SCD1)

**TODO:** Build from `ec_orders.promo_code` distinct values + the BTS campaign metadata. Carry `promotion_channel_scope` (which channels the promo applies to).

In [ ]:
# TODO M5: dim_promotion

## dim_store (M4, SCD1)

**TODO:** From `pos_stores`. Include `campaign_zone_activation_date` (per-store BTS dedicated counter activation; you may need to derive this from inventory events). 20 stores.

In [ ]:
# TODO M4: dim_store

## dim_seller (M6, SCD1, carries seller_risk_score)

**TODO:** From `silver_sellers` joined to `silver_seller_trust_score` (your T10 output). Carries `seller_risk_score` (0.00–1.00) and FK to `dim_seller_risk_tier`.

In [ ]:
# TODO M6: dim_seller

## dim_product (M3, SCD2 — most complex of M3's work)

**SCD2 mechanics:**
- For each `canonical_product_key`, detect attribute changes between Bronze ingestion versions (use `_ingestion_timestamp`)
- Emit one row per version with `valid_from`, `valid_to`, `is_current`
- `valid_to` of v_n = `valid_from` of v_(n+1) − 1 second; latest version `valid_to = '9999-12-31'`
- `is_current = TRUE` only on latest version per natural key

Window pattern:
```python
w = Window.partitionBy('canonical_product_key').orderBy('_ingestion_timestamp')
df = df.withColumn('valid_from', F.col('_ingestion_timestamp'))
df = df.withColumn('valid_to', F.lead('_ingestion_timestamp').over(w) - F.expr('INTERVAL 1 SECOND'))
df = df.withColumn('is_current', F.col('valid_to').isNull())
df = df.fillna({'valid_to': '9999-12-31 23:59:59'})
df = df.withColumn('version_number', F.row_number().over(w))
```

Carry `product_master_confidence` from T5.

In [ ]:
# TODO M3: dim_product (SCD2 implementation)

## dim_customer (M2, SCD2 + identity bridge — most complex dim overall)

Same SCD2 mechanics as `dim_product` but on `customer_master_key`.

**Identity bridge component:**
An additional table `dim_customer_identity_bridge` may be needed to map multiple natural-key sources (loyalty_id, email, phone, anonymous) → one canonical `customer_master_key`. Members at confidence 0.70–0.89 carry a probabilistic match audit trail.

Carry `identity_confidence_score` and `match_pass_used` from your T4 output.

In [ ]:
# TODO M2: dim_customer (SCD2 + identity bridge)

## Acceptance for the Gold dim layer

Lead checks:
1. All 13 dims exist in `NEXAMART_GOLD`
2. Every dim has unique `<dim>_key` (no duplicates within is_current=TRUE for SCD2)
3. Mandatory anomaly columns carried from Silver
4. SCD2 dims have correctly chained `valid_from`/`valid_to`/`is_current`/`version_number`
5. `dim_date` covers the whole 2024 calendar year
6. `dim_seller_risk_tier` has 5 tier rows; `dim_step` has 5 step rows; `dim_channel` has 5 channel rows